In [56]:
import pandas as pd
import csv
import pandas_gbq
import time
from datetime import datetime
import os
from google.oauth2 import service_account
from google.cloud import bigquery

In [57]:
CREDS = '../converge-database-0331482f2ee5.json'

In [58]:
client = bigquery.Client.from_service_account_json(json_credentials_path=CREDS)

In [59]:
seriatim = pd.DataFrame()
premium = pd.DataFrame()
withdrawals = pd.DataFrame()
terminated = pd.DataFrame()
annuitized = pd.DataFrame()

In [62]:
file="I:/New Structure/Actuarial New/Database/Heartland/202605/HNL Converge Report 20260531.xlsx"

In [63]:
seriatim = pd.read_excel(file, dtype="str", sheet_name = "Seriatim")

In [64]:
premium = pd.read_excel(file, dtype="str", sheet_name = "Premiums")

In [65]:
withdrawals = pd.read_excel(file, dtype="str", sheet_name = "Withdrawals")

In [66]:
terminated = pd.read_excel(file, dtype="str", sheet_name = "Terminated Policies")

In [67]:
annuitized = pd.read_excel(file, dtype="str", sheet_name = "Annuitized Policies")

In [68]:
seriatim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2720 entries, 0 to 2719
Data columns (total 67 columns):
 #   Column                                       Non-Null Count  Dtype 
---  ------                                       --------------  ----- 
 0   Inforce Date                                 2720 non-null   object
 1   Policy Number                                2720 non-null   object
 2   Plan                                         2720 non-null   object
 3   Issue Date                                   2720 non-null   object
 4   Renewal Date                                 2720 non-null   object
 5   Guarantee Period End Date                    2720 non-null   object
 6   Maturity Date                                2720 non-null   object
 7   Date Approved                                2720 non-null   object
 8   Issue State                                  2720 non-null   object
 9   Issue Age                                    2720 non-null   object
 10  Gender      

In [69]:
# Lowercasing the headers and removing the spaces between them
seriatim.columns = seriatim.columns.str.strip()
seriatim.columns = seriatim.columns.map(str.lower)
seriatim.columns = seriatim.columns.map(lambda x : x.replace(" ", "_"))
seriatim.columns = seriatim.columns.map(lambda x : x.replace("/", "_"))
seriatim.columns = seriatim.columns.map(lambda x : x.replace("+", "_plus"))

In [70]:
seriatim = seriatim.rename(columns={"additional_premiums.1": "additional_premiums_mtd", "inforce_date" : "set_month", "additional_premiums_total" : "additional_premiums_mtd", "stat_reserves": "stat_reserve", "tax_reserves":"tax_reserve" })

In [71]:
seriatim.set_month=seriatim.set_month.astype("datetime64[ns]")
seriatim['set_month']= seriatim['set_month'].dt.strftime('%Y%m')
set_month = seriatim.set_month[0]

In [72]:
seriatim = seriatim.astype({"issue_date":"datetime64[ns]","renewal_date":"datetime64[ns]", "guarantee_period_end_date" :"datetime64[ns]","maturity_date":"datetime64[ns]", "date_approved" :"datetime64[ns]","issue_age" :"int64",
                     "purchase_price" :"float64","additional_premiums" : "float64","total_premiums" :"float64", "bom_fund_value" : "float64", "interest_credited" : "float64",
                     "bonus_credited" : "float64", "additional_premiums_mtd" : "float64", "rmd_withdrawals" : "float64", "free_interest_credit_withdrawals" : "float64",
                     "freelook_withdrawals" : "float64","cancellation_withdrawals" :"float64","death_benefit" : "float64",
                     "enhanced_benefit_withdrawals" : "float64", "free_partial_withdrawals" : "float64", "partial_withdrawal_with_sc" : "float64",
                      "full_surrender_withdrawals" : "float64", "surrender_charges" : "float64", "expense_charges" : "float64", "eom_fund_value" : "float64",
                     "cumulative_interest_credited" : "float64","cumulative_bonus_credited" :"float64","cumulative_additional_premiums" : "float64",
                     "cumulative_rmd_withdrawals" : "float64", "cumulative_free_interest_credit_withdrawals" : "float64", "cumulative_freelook_withdrawals" : "float64",
                     "cumulative_cancellation_withdrawals" : "float64", "cumulative_death_benefit" : "float64", "cumulative_enhanced_benefit_withdrawals" : "float64", "cumulative_free_partial_withdrawals" : "float64",
                     "cumulative_partial_withdrawal_with_sc" : "float64","cumulative_full_surrender_withdrawals" :"float64","cumulative_surrender_charges" : "float64",
                     "cumulative_expense_charges" : "float64", "gmsv" : "float64", "free_partial_withdrawal_rider" : "int64", "death_benefit_rider" : "int64",
                        "enhanced_benefit_rider" : "int64", "bonus_crediting_rider" : "int64", "year_1_int_rate" : "float64", "year_2_plus_int_rate" : "float64",
                        "guaranteed_minimum_crediting_rate" : "float64", "snfl_crediting_rate" : "float64", "renewal_indicator" : "int64", "surrender_value_w_o_mva" : "float64",
                        "surrender_value_w_mva" : "float64", "stat_reserve" : "float64", "tax_reserve" : "float64", "quota_share" : "float64", "set_month" : "object"
               })

In [73]:
seriatim = seriatim.iloc[:, 0:63]

In [74]:
seriatim

,set_month,policy_number,plan,issue_date,renewal_date,guarantee_period_end_date,maturity_date,date_approved,issue_state,issue_age,...,guaranteed_minimum_crediting_rate,snfl_crediting_rate,myga_dep_type,renewal_indicator,surrender_value_w_o_mva,surrender_value_w_mva,stat_reserve,tax_reserve,quota_share,plangroup
0,202605,HN40001006P1,MYGE21-IRA,2024-01-23,2024-01-23,2027-01-22,2035-01-23,2023-12-13,IL,89,...,0.03,0.030,MYGA,0,52808.40,52801.49,56785.25,52808.40,0.95,MYG03
1,202605,HN40001007P1,MYGE21-IRA,2024-02-06,2024-02-06,2031-02-05,2061-02-06,2024-01-02,IL,63,...,0.03,0.030,MYGA,0,161205.94,159788.90,177208.86,161205.94,0.95,MYG07
2,202605,HN40001008P1,MYGE21,2024-01-18,2024-01-18,2029-01-17,2041-01-18,2024-01-11,IL,83,...,0.03,0.030,MYGA,0,0.00,0.00,0.00,0.00,0.95,MYG05
3,202605,HN40001009P1,MYGE21,2024-01-12,2024-01-12,2031-01-11,2055-01-12,2024-01-08,IL,69,...,0.03,0.030,MYGA,0,106387.54,104707.29,116889.38,106387.55,0.95,MYG07
4,202605,HN40001010P1,MYGE21,2024-01-30,2024-01-30,2027-01-29,2048-01-30,2024-01-22,IL,76,...,0.03,0.030,MYGA,0,35642.41,35697.71,38352.90,35642.41,0.95,MYG03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2715,202605,HN4F003744P1,MYGE24,2025-08-18,2025-08-18,2028-08-17,2051-08-18,2025-08-14,IN,74,...,0.01,0.028,MYGA,0,94414.25,93993.67,102582.45,95206.77,0.95,MYG03
2716,202605,HN4F003748P1,MYGE24-FL,2025-08-22,2025-08-22,2028-08-21,2064-08-22,2025-08-19,FL,61,...,0.01,0.028,MYGA,0,47182.87,46829.39,51257.48,47572.06,0.95,MYG03
2717,202605,HN4F003753P1,MYGE24,2025-08-26,2025-08-26,2030-08-25,2056-08-26,2025-08-21,IL,69,...,0.01,0.028,MYGA,0,14214.56,14020.24,15710.75,14466.65,0.95,MYG05
2718,202605,HN4F003754P1,MYGE24,2025-08-28,2025-08-28,2030-08-27,2057-08-28,2025-08-21,IL,68,...,0.01,0.028,MYGA,0,14210.43,14034.78,15707.59,14463.00,0.95,MYG05


In [75]:
seriatim.to_gbq("converge-database.heartland.seriatim",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [76]:
premium.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Policy Number       1 non-null      object
 1   PLAN                1 non-null      object
 2   Issue Date          1 non-null      object
 3   Transaction Date    1 non-null      object
 4   BOM Fund Value      1 non-null      object
 5   Initial Premium     1 non-null      object
 6   Additional Premium  1 non-null      object
 7   Total Premium       1 non-null      object
 8   Renewal Premium     1 non-null      object
 9   Initial Cede        1 non-null      object
 10  Renewal Cede        1 non-null      object
 11  Initial Commission  1 non-null      object
 12  Renewal Commission  1 non-null      object
 13  Quota Share         1 non-null      object
 14  PlanGroup           1 non-null      object
dtypes: object(15)
memory usage: 248.0+ bytes


In [77]:
#Lowercasing the headers and removing the spaces between them
premium.columns = premium.columns.str.strip()
premium.columns = premium.columns.map(str.lower)
premium.columns = premium.columns.map(lambda x : x.replace(" ", "_"))
premium.columns = premium.columns.map(lambda x : x.replace("/", "_"))

In [78]:
premium = premium.astype({"issue_date" : "datetime64[ns]", "transaction_date" : "datetime64[ns]", "bom_fund_value" : "float64", "initial_premium" : "float64", 
                          "additional_premium" : "float64", "total_premium" : "float64", "renewal_premium" : "float64", "renewal_cede" : "float64",
                         "initial_cede": "float64", "initial_commission" : "float64", "renewal_commission" : "float64", "quota_share" :"float64"})

In [79]:
premium = premium.iloc[:,0:15]

In [80]:
premium['set_month'] = set_month

In [81]:
premium

,policy_number,plan,issue_date,transaction_date,bom_fund_value,initial_premium,additional_premium,total_premium,renewal_premium,initial_cede,renewal_cede,initial_commission,renewal_commission,quota_share,plangroup,set_month
0,HN40001526P1,MYGE21,2024-09-30,2026-05-27,16278.45,0.0,0.0,0.0,0.0,0.0,0.0,-131.25,0.0,0.95,MYG03,202605


In [82]:
premium.to_gbq("converge-database.heartland.premium",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [83]:
withdrawals

,Policy Number,PLAN,Issue Date,Transaction Date,BOM Fund Value,Withdrawal Type,Withdrawal Amount,Surrender Charge,EOM Fund Value,EOGP,Quota Share,PlanGroup,Unnamed: 12
0,HN40001125P1,MYGE21-IRA,2024-06-12 00:00:00,2026-05-01 00:00:00,40801.07,Free Partial Withdrawals,2169,0,38820.47,N,0.95,MYG05,MYG05NFree Partial Withdrawals
1,HN40001073P1,MYGE21,2024-05-01 00:00:00,2026-05-01 00:00:00,351516.79,Free Partial Withdrawals,1648.4,0,351571.1,N,0.95,MYG05,MYG05NFree Partial Withdrawals
2,HN40001120P1,MYGE21,2024-05-29 00:00:00,2026-05-01 00:00:00,122790.74,Free Partial Withdrawals,556.71,0,122828.78,N,0.95,MYG05,MYG05NFree Partial Withdrawals
3,HN40001124P1,MYGE21-IRA,2024-06-05 00:00:00,2026-05-01 00:00:00,25443.95,RMD Withdrawals,221.73,-8.44,25345.88,N,0.95,MYG05,MYG05NRMD Withdrawals
4,HN40001145P1,MYGE21,2024-05-30 00:00:00,2026-05-01 00:00:00,45192.95,Free Partial Withdrawals,199.76,0,45203.71,N,0.95,MYG03,MYG03NFree Partial Withdrawals
...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,HN4F001866P1,MYGE21-FL,2024-08-27 00:00:00,2026-05-28 00:00:00,100086.29,Free Partial Withdrawals,504.35,0,100043.95,N,0.95,MYG03,MYG03NFree Partial Withdrawals
242,HN4F002696P1,MYGE21-IRA,2025-01-28 00:00:00,2026-05-28 00:00:00,208475.32,Free Partial Withdrawals,965,0,208506.98,N,0.95,MYG10,MYG10NFree Partial Withdrawals
243,HN4F002715P1,MYGE21-FL,2024-11-01 00:00:00,2026-05-28 00:00:00,200051.82,Free Partial Withdrawals,838.76,0,200079.45,N,0.95,MYG03,MYG03NFree Partial Withdrawals
244,HN4F003717P1,MYGE24,2025-07-28 00:00:00,2026-05-28 00:00:00,5091.59,Free Partial Withdrawals,113.04,0,5002.22,N,0.95,MYG07,MYG07NFree Partial Withdrawals


In [84]:
withdrawals.columns = withdrawals.columns.str.strip()
withdrawals.columns = withdrawals.columns.map(str.lower)
withdrawals.columns = withdrawals.columns.map(lambda x : x.replace(" ", "_"))
withdrawals.columns = withdrawals.columns.map(lambda x : x.replace("/", "_"))

In [85]:
withdrawals

,policy_number,plan,issue_date,transaction_date,bom_fund_value,withdrawal_type,withdrawal_amount,surrender_charge,eom_fund_value,eogp,quota_share,plangroup,unnamed:_12
0,HN40001125P1,MYGE21-IRA,2024-06-12 00:00:00,2026-05-01 00:00:00,40801.07,Free Partial Withdrawals,2169,0,38820.47,N,0.95,MYG05,MYG05NFree Partial Withdrawals
1,HN40001073P1,MYGE21,2024-05-01 00:00:00,2026-05-01 00:00:00,351516.79,Free Partial Withdrawals,1648.4,0,351571.1,N,0.95,MYG05,MYG05NFree Partial Withdrawals
2,HN40001120P1,MYGE21,2024-05-29 00:00:00,2026-05-01 00:00:00,122790.74,Free Partial Withdrawals,556.71,0,122828.78,N,0.95,MYG05,MYG05NFree Partial Withdrawals
3,HN40001124P1,MYGE21-IRA,2024-06-05 00:00:00,2026-05-01 00:00:00,25443.95,RMD Withdrawals,221.73,-8.44,25345.88,N,0.95,MYG05,MYG05NRMD Withdrawals
4,HN40001145P1,MYGE21,2024-05-30 00:00:00,2026-05-01 00:00:00,45192.95,Free Partial Withdrawals,199.76,0,45203.71,N,0.95,MYG03,MYG03NFree Partial Withdrawals
...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,HN4F001866P1,MYGE21-FL,2024-08-27 00:00:00,2026-05-28 00:00:00,100086.29,Free Partial Withdrawals,504.35,0,100043.95,N,0.95,MYG03,MYG03NFree Partial Withdrawals
242,HN4F002696P1,MYGE21-IRA,2025-01-28 00:00:00,2026-05-28 00:00:00,208475.32,Free Partial Withdrawals,965,0,208506.98,N,0.95,MYG10,MYG10NFree Partial Withdrawals
243,HN4F002715P1,MYGE21-FL,2024-11-01 00:00:00,2026-05-28 00:00:00,200051.82,Free Partial Withdrawals,838.76,0,200079.45,N,0.95,MYG03,MYG03NFree Partial Withdrawals
244,HN4F003717P1,MYGE24,2025-07-28 00:00:00,2026-05-28 00:00:00,5091.59,Free Partial Withdrawals,113.04,0,5002.22,N,0.95,MYG07,MYG07NFree Partial Withdrawals


In [86]:
withdrawals = withdrawals.iloc[:,0:12]

In [87]:
withdrawals = withdrawals.astype({"issue_date" : "datetime64[ns]", "transaction_date" : "datetime64[ns]", "bom_fund_value" : "float64", 
                          "withdrawal_amount" : "float64", "surrender_charge" : "float64", "eom_fund_value" : "float64", "quota_share" :"float64"})

In [88]:
withdrawals['set_month'] = set_month

In [89]:
withdrawals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246 entries, 0 to 245
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   policy_number      246 non-null    object        
 1   plan               246 non-null    object        
 2   issue_date         246 non-null    datetime64[ns]
 3   transaction_date   246 non-null    datetime64[ns]
 4   bom_fund_value     246 non-null    float64       
 5   withdrawal_type    246 non-null    object        
 6   withdrawal_amount  246 non-null    float64       
 7   surrender_charge   246 non-null    float64       
 8   eom_fund_value     246 non-null    float64       
 9   eogp               246 non-null    object        
 10  quota_share        246 non-null    float64       
 11  plangroup          246 non-null    object        
 12  set_month          246 non-null    object        
dtypes: datetime64[ns](2), float64(5), object(6)
memory usage: 25.1+ K

In [90]:
withdrawals.to_gbq("converge-database.heartland.withdrawals",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [91]:
terminated

,Policy Number,PLAN,Issue Date,Termination Date,Cumulative Premium,Cumulative Interest Earned,Withdrawal Type,Withdrawal Amount,Cumulative Surrender Charge,Fund Value on Termination,EOGP,Quota Share,PlanGroup
0,HN40001526P1,MYGE21,2024-09-30 00:00:00,2026-05-27 00:00:00,15000,1140.84,Death Benefit,16140.84,0,0,N,0.95,MYG03
1,HN4F001075P1,MYGE21-IRA,2024-03-22 00:00:00,2026-05-06 00:00:00,75335.72,-2036.62,Full Surrender Withdrawals,85186.31999999999,5943.61,0,N,0.95,MYG05


In [92]:
terminated.columns = terminated.columns.str.strip()
terminated.columns = terminated.columns.map(str.lower)
terminated.columns = terminated.columns.map(lambda x : x.replace(" ", "_"))
terminated.columns = terminated.columns.map(lambda x : x.replace("/", "_"))

In [93]:
terminated = terminated.astype({"issue_date" : "datetime64[ns]", "termination_date" : "datetime64[ns]", "cumulative_premium" : "float64", 
                          "cumulative_interest_earned" : "float64", "withdrawal_amount" : "float64", "cumulative_surrender_charge" : "float64", "fund_value_on_termination" :"float64", 'quota_share' : 'float64'})

In [94]:
terminated['set_month'] = set_month

In [95]:
terminated.to_gbq("converge-database.heartland.terminated",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [96]:
annuitized

,Policy Number,PLAN,Issue Date,Transaction Date,Initial Annuitized Account Value,Current Annuitized Account Value,Annuitized Payment,Annuitization Method,Annuitization Interest,Annuitization Mortality Table,Annuitized Reserves,Quota Share,PlanGroup,Pymt NILC,Pymt ILC,Unnamed: 15,Intial Rsv NILC,Intial Rsv ILC
0,HN4S001067P1,ANWOL,2024-06-25 00:00:00,NaN,NaN,11759.68,0,Certain,-0.003933404204872204,NaN,11759.68,0.95,MYG03,0,0,NaN,0,0


In [97]:
annuitized.columns = annuitized.columns.str.strip()
annuitized.columns = annuitized.columns.map(str.lower)
annuitized.columns = annuitized.columns.map(lambda x : x.replace(" ", "_"))
annuitized.columns = annuitized.columns.map(lambda x : x.replace("/", "_"))

In [98]:
annuitized = annuitized.rename(columns={"current_annuitized_account_value" : "annuitized_account_value"})

In [99]:
annuitized = annuitized.astype({"issue_date" : "datetime64[ns]", "transaction_date" : "datetime64[ns]", "annuitized_account_value" : "float64", 
                          "annuitized_payment" : "float64", "annuitization_method" : "object", "annuitization_interest" : "float64", "annuitized_reserves" :"float64", 'quota_share' : 'float64'})

In [100]:
annuitized= annuitized.iloc[:, 0:13]

In [101]:
annuitized = annuitized.drop({"annuitization_mortality_table"}, axis=1)

In [102]:
annuitized['set_month'] = set_month

In [103]:
annuitized.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 13 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   policy_number                     1 non-null      object        
 1   plan                              1 non-null      object        
 2   issue_date                        1 non-null      datetime64[ns]
 3   transaction_date                  0 non-null      datetime64[ns]
 4   initial_annuitized_account_value  0 non-null      object        
 5   annuitized_account_value          1 non-null      float64       
 6   annuitized_payment                1 non-null      float64       
 7   annuitization_method              1 non-null      object        
 8   annuitization_interest            1 non-null      float64       
 9   annuitized_reserves               1 non-null      float64       
 10  quota_share                       1 non-null      floa

In [104]:
annuitized.to_gbq("converge-database.heartland.annuitized",
                 if_exists='append',
                  table_schema=None,
                 project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [ ]:
# query_to_fix ='UPDATE `heartland.seriatim`\
# SET plangroup = CONCAT(LEFT(plangroup,3), "E", right(plangroup,2))\
# WHERE set_month ='202409'\

In [105]:
tableName = ['seriatim','premium', 'withdrawals', 'terminated', 'annuitized']

for j in tableName:
    query = f'''
        UPDATE `heartland.{j}` SET plangroup=CONCAT(LEFT(plangroup,3), "E", right(plangroup,2))\
        WHERE set_month="{set_month}"
        '''
    print(query)
    CREDS = '../converge-database-0331482f2ee5.json'
    client = bigquery.Client.from_service_account_json(json_credentials_path=CREDS)
    result = client.query(query)
    print(result.result())



        UPDATE `heartland.seriatim` SET plangroup=CONCAT(LEFT(plangroup,3), "E", right(plangroup,2))        WHERE set_month="202605"
        

        UPDATE `heartland.premium` SET plangroup=CONCAT(LEFT(plangroup,3), "E", right(plangroup,2))        WHERE set_month="202605"
        

        UPDATE `heartland.withdrawals` SET plangroup=CONCAT(LEFT(plangroup,3), "E", right(plangroup,2))        WHERE set_month="202605"
        

        UPDATE `heartland.terminated` SET plangroup=CONCAT(LEFT(plangroup,3), "E", right(plangroup,2))        WHERE set_month="202605"
        

        UPDATE `heartland.annuitized` SET plangroup=CONCAT(LEFT(plangroup,3), "E", right(plangroup,2))        WHERE set_month="202605"
        


In [106]:
reported_date_q= f'''
UPDATE `heartland.seriatim`  s
SET s.reported_date = (SELECT MIN(s2.set_month) FROM `heartland.seriatim` s2 WHERE s2.policy_number = s.policy_number)
WHERE s.reported_date is NULL'''
job = client.query(reported_date_q) 

In [107]:
import sys
sys.path.append('../actuarial-pipelines/reconciliations/heartland')

from reconciliation import run_reconciliation

run_reconciliation(set_month)

Running reconciliation for:  202605
SELECT SUM(total_premium)*0.95 as premium_MYGE03 FROM `heartland.premium`                WHERE set_month ="202605" AND plangroup ="MYGE03"
{'set_month': '202605', 'plan_group': 'MYGE03', 'type': 'total_premium', 'total': 0.0}
SELECT SUM(renewal_premium)*0.95 as premium_MYGE03 FROM `heartland.premium`                WHERE set_month ="202605" AND plangroup ="MYGE03"
{'set_month': '202605', 'plan_group': 'MYGE03', 'type': 'renewal_premium', 'total': 0.0}
SELECT SUM(total_premium)*0.95 as premium_MYGE05 FROM `heartland.premium`                WHERE set_month ="202605" AND plangroup ="MYGE05"
{'set_month': '202605', 'plan_group': 'MYGE05', 'type': 'total_premium', 'total': None}
SELECT SUM(renewal_premium)*0.95 as premium_MYGE05 FROM `heartland.premium`                WHERE set_month ="202605" AND plangroup ="MYGE05"
{'set_month': '202605', 'plan_group': 'MYGE05', 'type': 'renewal_premium', 'total': None}
SELECT SUM(total_premium)*0.95 as premium_MYGE07 F

In [108]:
# Add directory containing LDTI.py
sys.path.append('../actuarial-pipelines/ldti')

# Import the function
from LDTI import main_query_run

# Trigger the AVRF analysis
main_query_run("Heartland")

Starting LDTI run for : Heartland
starting Heartland LDTI

       WITH min_set_month AS(
          select policy_number, plangroup, MIN(set_month) as first_set_month,
          LEFT(reported_date,4) AS report_year,
          FROM `heartland.seriatim`
          GROUP by policy_number, plangroup,report_year
        ),
        inforce_list AS (
          select set_month, plangroup, LEFT(reported_date,4) as report_year, COUNT(distinct policy_number) as ct_inforce
          FROM `heartland.seriatim`
          WHERE eom_fund_value>0
          GROUP by set_month, plangroup, report_year
        )
        select i.set_month, i.plangroup, i.report_year,i.ct_inforce, count(m.policy_number) as new_issued_policies
        FROM inforce_list i
        LEFT JOIN min_set_month m
          ON m.first_set_month = i.set_month AND m.report_year = i.report_year AND i.plangroup = m.plangroup
        GROUP BY i.set_month, i.plangroup, i.report_year, i.ct_inforce
        ORDER BY i.set_month, i.plangroup, i.r

In [109]:
# Add directory containing LDTI.py
sys.path.append('../actuarial-pipelines/avrf/kskj')

# Import the function
from avrf import run_avrf_analysis

# Trigger the AVRF analysis
run_avrf_analysis(set_month, 'heartland')

AVRF analysis starting for heartland set_month 202605
Total absolute difference in AVRF AV: 730.70
Results saved to Query Results/AVRF/AVRF_heartland_202605.xlsx


,policy_number,issue_date,beginning_fund_value,premium,interest_credited,bonus_credited,rmd_withdrawals,free_interest_credit_withdrawals,freelook_withdrawals,cancellation_withdrawals,...,beginning_reserve_stat,end_reserve_stat,new_policy_check,dropped_policy_check,plan,plangroup,inflow,outflow,exp_av,diff
0,HN40001001P1,2023-11-14,10850.3205,0.0,50.5590,0.0,0.0,0.0,0.0,0.0,...,10860.2005,10906.2755,0,0,MYGE21,MYGE03,50.5590,0.0,10900.8795,1.818989e-12
1,HN40001003P1,2024-01-04,131758.5305,0.0,640.9555,0.0,0.0,0.0,0.0,0.0,...,133319.3805,133912.9025,0,0,MYGE21,MYGE05,640.9555,0.0,132399.4860,0.000000e+00
2,HN40001004P1,2024-01-04,131758.5305,0.0,640.9555,0.0,0.0,0.0,0.0,0.0,...,133334.3145,133926.9150,0,0,MYGE21,MYGE05,640.9555,0.0,132399.4860,0.000000e+00
3,HN40001006P1,2024-01-23,53693.8480,0.0,250.2110,0.0,0.0,0.0,0.0,0.0,...,53707.4805,53945.9875,0,0,MYGE21-IRA,MYGE03,250.2110,0.0,53944.0590,7.275958e-12
4,HN40001007P1,2024-02-06,163875.5415,0.0,797.1925,0.0,0.0,0.0,0.0,0.0,...,167602.9900,168348.4170,0,0,MYGE21-IRA,MYGE07,797.1925,0.0,164672.7340,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2717,HN4F003744P1,2025-08-18,98167.3855,0.0,396.9385,0.0,0.0,0.0,0.0,0.0,...,97026.7775,97453.3275,0,0,MYGE24,MYGE03,396.9385,0.0,98564.3240,0.000000e+00
2718,HN4F003748P1,2025-08-22,49058.4750,0.0,198.3790,0.0,0.0,0.0,0.0,0.0,...,48481.2550,48694.6060,0,0,MYGE24-FL,MYGE03,198.3790,0.0,49256.8540,0.000000e+00
2719,HN4F003753P1,2025-08-26,14771.7495,0.0,67.6210,0.0,0.0,0.0,0.0,0.0,...,14859.3585,14925.2125,0,0,MYGE24,MYGE05,67.6210,0.0,14839.3705,1.818989e-12
2720,HN4F003754P1,2025-08-28,14767.4555,0.0,67.6020,0.0,0.0,0.0,0.0,0.0,...,14856.4135,14922.2105,0,0,MYGE24,MYGE05,67.6020,0.0,14835.0575,-1.818989e-12
